In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdata202611")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/actors.csv"

In [0]:
actors_schema = StructType(fields=[StructField("personId", IntegerType(), False),
                                  StructField("personname", StringType(), True),
                                  StructField("lastname", StringType(), True)
])

In [0]:
actors_df = spark.read \
            .option("header", True) \
            .schema(actors_schema) \
            .csv(ruta)

In [0]:
actors_with_timestamp_df = actors_df.withColumn("ingestion_date", current_timestamp())

In [0]:
actors_selected_df = actors_with_timestamp_df.select(col('personId').alias('person_Id'), 
                                                   col('personname').alias('person_name'),
                                                   col('lastname'), col('lastname'))

In [0]:
actors_selected_df.toDF('person_Id', 'person_name', 'lastname', 'lastname_dup').drop('lastname_dup') \
    .write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalogo}.{esquema}.actors')